<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/JS06-ANN/praktikum_6_Micro_Spotify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install annoy

!pip install faiss-cpu
#!pip install faiss-gpu

!pip install hnswlib

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Path to dataset files: /kaggle/input/spotify-songs-with-attributes-and-lyrics


In [7]:
# Otomatis cari lokasi dataset dari kagglehub
import kagglehub
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")

csv_path = os.path.join(path, "songs_with_attributes_and_lyrics.csv")
df = pd.read_csv(csv_path)

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.


In [8]:
import os
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
csv_path = os.path.join(path, "songs_with_attributes_and_lyrics.csv")
df = pd.read_csv(csv_path)

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
# FIX: Cukup query 1 lagu pertama (X_scaled[:1])
dist_exact, idx_exact = nn.kneighbors(X_scaled[:1])
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)

# FIX: Cukup query 1 lagu pertama (Hapus list comprehension seluruh data)
idx_annoy = [index_annoy.get_nns_by_vector(X_scaled[0], k)]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)

# FIX: Cukup query 1 lagu pertama (Hapus baris yang memanggil X_scaled tanpa [:1])
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled[:1], k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])

# FIX: Hapus parameter nlist= dan metric=
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10

# FIX: Cukup query 1 lagu pertama
dist_faiss, idx_faiss = index_faiss.search(X_scaled[:1], k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------------------
# Tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")

Exact NN done in 0.094 s
Annoy done in 17.285 s
HNSW done in 239.309 s
FAISS IVF done in 0.423 s

Top-5 neighbors for first song:
Exact NN: [     0 394553 764272 837727 749223]
Annoy:    [0, 764272, 749223, 625227, 833164]
HNSW:     [     0 394553 764272 837727 749223]
FAISS:    [     0 394553 764272 837727 749223]


1. Analisis Waktu Eksekusi (Kecepatan)
Exact NN (0.094 s):

Menjadi yang sangat cepat pada pengujian ini karena hanya melakukan pencarian brute-force untuk 1 sampel query. Namun, metode ini tidak membuat struktur indeks khusus di awal, sehingga tidak fleksibel untuk skala query yang besar.

FAISS IVF (0.423 s):

Algoritma paling efisien dan cepat. FAISS berhasil membangun indeks Inverted File Index (IVF) dan menyelesaikan pencarian dalam waktu di bawah 1 detik. Sangat optimal untuk pencarian vektor berskala jutaan data.

Annoy (17.285 s):

Membutuhkan waktu lebih lama saat membangun pohon (tree construction) karena iterasi penambahan item dilakukan satu per satu di level Python (for i, v in enumerate(X_scaled)).

HNSW (239.309 s / ~4 menit):

Membutuhkan waktu paling lama. Hal ini disebabkan oleh proses pembangunan graf hierarkis (Hierarchical Navigable Small World) dengan parameter ef_construction=200 dan M=16 yang sangat intensif secara komputasi untuk 1,8 juta data.

2. Analisis Hasil Pencarian (Akurasi / Top-5 Neighbors)
Hasil indeks tetangga terdekat dari lagu pertama:

Exact NN (Baseline/Ground Truth): [0, 394553, 764272, 837727, 749223]

HNSW: [0, 394553, 764272, 837727, 749223] (Akurasi 100%)

FAISS IVF: [0, 394553, 764272, 837727, 749223] (Akurasi 100%)

Annoy: [0, 764272, 749223, 625227, 833164] (Akurasi ~40% / Approximate)

Temuan Utama:

HNSW & FAISS IVF menghasilkan kandidat lagu yang persis sama (identical) dengan hasil pencarian Exact NN. Ini membuktikan bahwa kedua algoritma ini mampu mempertahankan recall/akurasi yang sangat tinggi tanpa mengorbankan kecepatan.

Annoy meleset pada beberapa tetangga terdekat (hanya menemukan lagu indeks 0, 764272, dan 749223). Ini menunjukkan sifat Approximate Nearest Neighbor (ANN) dari Annoy yang membagi ruang dengan hyperplane acak, sehingga ada potensi tetangga terdekat berada di cabang pohon (tree branch) yang berbeda.

3. Kesimpulan & Rekomendasi
FAISS IVF adalah pilihan terbaik dan paling seimbang untuk dataset berukuran besar seperti Spotify (1,8M data). FAISS menawarkan waktu pembangunan indeks terbilang instant (0.4 detik) dengan akurasi 100% tepat sama seperti Exact NN.

HNSW memiliki akurasi yang sangat baik, namun memiliki trade-off waktu build index yang sangat lama.

Annoy cocok untuk skenario sederhana dengan jumlah data menengah, tetapi kurang presisi jika dibandingkan dengan FAISS/HNSW pada fitur numerik Spotify.